# RAG

In [5]:
import hnswlib
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Initialize Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")
chunks = ["Chunk 1 text...", "Chunk 2 text...", "..."]
embeddings = model.encode(chunks, normalize_embeddings=True)

# 2. Setup Index Parameters
dim = embeddings.shape[1]
num_elements = len(embeddings)

# Initialize hnswlib index
# Space options: 'l2', 'cosine', or 'ip' (Inner Product)
p = hnswlib.Index(space='cosine', dim=dim) 

# Initializing index - capacity must be known beforehand
# M: Number of bi-directional links created for every new element. Higher M increases accuracy but uses more memory.
# ef_construction: Depth of the search during index construction. Higher values improve index quality but slow down the build.
p.init_index(max_elements=num_elements, ef_construction=200, M=16)

# 3. Add items to index
p.add_items(embeddings, np.arange(num_elements))

# 4. Save for later use
p.save_index("dat255_index.bin")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7467.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def retrieve_context(query, k=3):
    # Embed the query
    query_embedding = model.encode([query], normalize_embeddings=True)
    
    # query the system
    # ef (ef_search) controls the search speed/accuracy trade-off
    p.set_ef(50) 
    labels, distances = p.knn_query(query_embedding, k=k)
    
    return [chunks[i] for i in labels]

# Example usage
query = "What is the capital of France?"
context = retrieve_context(query, k=3)
print(context)

TypeError: only integer scalar arrays can be converted to a scalar index

In [8]:
import os
import hnswlib
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Initialize SOTA local embedding model
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def load_data_from_folder(folder_path):
    documents = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".txt") or file.endswith(".md"):
                with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                    documents.append({
                        "text": f.read(),
                        "source": file
                    })
    return documents

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6670.99it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_chunks(documents, chunk_size=500, chunk_overlap=50):
    # chunk_overlap ensures context is preserved between boundaries
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    chunks_with_metadata = []
    for doc in documents:
        # Split the text
        split_texts = splitter.split_text(doc["text"])
        for i, text in enumerate(split_texts):
            chunks_with_metadata.append({
                "text": text,
                "source": doc["source"],
                "chunk_id": i
            })
    return chunks_with_metadata

In [13]:
# Create example data for testing
example_text = """
The Transformer architecture uses Self-Attention to weigh the importance of different words. 
Unlike RNNs, it processes all words in parallel. 
To keep track of word order, Positional Encodings are added to the input embeddings.
The Adam optimizer is commonly used for training because it adapts the learning rate for each parameter.
"""

# Test the pipeline
docs = [{"text": example_text, "source": "example_lecture.txt"}]
chunks = create_chunks(docs, chunk_size=100, chunk_overlap=20)

# 1. Embed chunks
chunk_texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)

# 2. Initialize hnswlib index
dim = embeddings.shape[1]
num_elements = len(embeddings)
p = hnswlib.Index(space='cosine', dim=dim) 
p.init_index(max_elements=num_elements, ef_construction=200, M=16)
p.add_items(embeddings, np.arange(num_elements))

# 3. Standalone Retrieval Test
query = "How do transformers handle word order?"
query_embedding = embed_model.encode([query], normalize_embeddings=True)

labels, distances = p.knn_query(query_embedding, k=2)
print(f"Query: {query}")
for idx in labels[0]:
    print(f"\n{chunks[int(idx)]['source']}")
    print(f"Retrieved Chunk: {chunks[int(idx)]['text']}")
# for idx in labels:
#     print(f"\n['source']]")
#     print(f"Retrieved Chunk: {chunks[idx]['text']}")

Query: How do transformers handle word order?

example_lecture.txt
Retrieved Chunk: The Transformer architecture uses Self-Attention to weigh the importance of different words.

example_lecture.txt
Retrieved Chunk: To keep track of word order, Positional Encodings are added to the input embeddings.
